In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# JTA Challenge — Demonstração e QA

Sistema de recomendação *agentic* para produtos Nintendo Switch.

Este notebook demonstra o sistema em funcionamento e documenta o **QA** pedido no enunciado e os três métodos de
recomendação implementados (co-ocorrência direta vs similaridade de cosseno vs NPMI).


## 1. Setup

In [ ]:
import os
import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")


## 2. Load e validação dos dados

Os dois ficheiros são lidos e a coerência entre eles é validada (os nomes de produtos no JSON têm de coincidir com os da matriz de co-ocorrência).

In [ ]:
from data_loader import load_all

products, matrix = load_all()
print(f"{len(products)} produtos carregados.")
print(f"Matriz de co-ocorrência: {matrix.shape[0]}x{matrix.shape[1]}")


In [ ]:
products["Super Mario Odyssey"]

## 3. Tools

Antes de envolver o LLM, validamos que cada tool funciona isoladamente.

### 3.1 Três métodos de recomendação

- **Co-ocorrência direta**: produtos comprados *no mesmo cabaz* (complementaridade).
- **Similaridade de cosseno**: produtos com *perfil de co-compra parecido* (afinidade/alternativa).
- **NPMI** (motor principal, Market Basket Analysis): mede se dois produtos co-ocorrem *acima do esperado por acaso*, corrigindo o viés de popularidade da co-ocorrência bruta. O cosseno serve de validação cruzada.

In [ ]:
from tools import get_cooccurring_products, get_similar_products, get_recommendations_npmi

print("Co-ocorrência direta (comprado junto com):")
for item in get_cooccurring_products("Super Mario Odyssey", top_n=5)["cooccurring"]:
    print(f"  {item['product']}: {item['times_bought_together']}")

print("\nSimilaridade de cosseno (perfil parecido):")
for item in get_similar_products("Super Mario Odyssey", top_n=5)["similar"]:
    print(f"  {item['product']}: {item['similarity']}")

print("\nNPMI (associação acima do acaso — motor principal):")
npmi_result = get_recommendations_npmi("Super Mario Odyssey", top_n=5)
for item in npmi_result["recommendations"]:
    print(f"  {item['product']}: NPMI={item['npmi']} (cosseno cross-check={item['cosine_crosscheck']})")

### 3.2 Filtros: loja, idade, exclusão de franchise

In [11]:
from tools import search_products, filter_by_store, exclude_franchise

# Jogos indicados para uma criança de 5 anos
jogos_5_anos = [p["name"] for p in search_products(category="Game", max_min_age=5)]
print("Apropriados até 5 anos:", jogos_5_anos)

# Excluir a franchise Super Mario
sem_mario = exclude_franchise(jogos_5_anos, "Super Mario")
print("Sem Super Mario:", sem_mario)

# Dos que sobram, quais estão na Store A
na_store_a = [p["name"] for p in filter_by_store(sem_mario, "Store_A")]
print("Disponíveis na Store A:", na_store_a)


Apropriados até 5 anos: ['Animal Crossing: New Horizons']
Sem Super Mario: ['Animal Crossing: New Horizons']
Disponíveis na Store A: ['Animal Crossing: New Horizons']


## 4. QA do sistema agentic completo

Agora o sistema end to end: o LLM recebe a query, decide que tools chamar, e executa-as.

### 4.1 Query não relacionada (exemplo do enunciado)

> *"I want a pepperoni pizza with extra cheese please."*

O sistema deve recusar educadamente, sem inventar produtos.

In [12]:
from agent import run_agent

resposta = run_agent("I want a pepperoni pizza with extra cheese please.", verbose=True)
print("\n>>> Resposta final:\n", resposta)



>>> Resposta final:
 Desculpe, mas só posso ajudar com recomendações de produtos Nintendo Switch. Se precisar de algo nessa categoria, estou aqui para ajudar!


### 4.2 Query complexa (exemplo do enunciado)

> *"I want to buy a game for my nephew, at Store A, who is 5 years old. We loved Super Mario Odyssey,
> but I cannot buy a game from this family as he already has all Super Mario games."*

Esta query exige encadear vários passos: identificar o produto de referência, encontrar parecidos,
excluir a franchise Super Mario, filtrar por idade (5 anos) e por loja (Store A).

In [13]:
query = ("I want to buy a game for my nephew, at Store A, who is 5 years old. "
         "We loved Super Mario Odyssey, but I cannot buy a game from this family "
         "as he already has all Super Mario games.")

resposta = run_agent(query, verbose=True)
print("\n>>> Resposta final:\n", resposta)


  [tool call] search_products({'max_min_age': 5})
  [tool result] [{'name': 'Nintendo Switch', 'category': 'Console', 'type': None, 'franchise': None, 'min_age': None, 'release_date': None, 'times_sold': 500000, 'url': 'https://www.nintendo.com/us/search/#q=Nintendo+Switch', 'store_breakdown': {'Store_A': 200000, 'Store_B': 150000, 'Store_C': 150000}}, {'name': 'Animal Crossing: New Horizons', 'category': 'Game', 'type': 'Simulation', 'franchise': 'Animal Crossing', 'min_age': 3, 'release_date': '2020-03-20', 'times_sold': 800000, 'url': 'https://www.nintendo.com/us/search/#q=Animal+Crossing%3A+New+Horizons', 'store_breakdown': {'Store_A': 500000, 'Store_B': 300000, 'Store_C': None}}, {'name': 'Nintendo Switch Pro Controller', 'category': 'Accessory', 'type': 'Controller', 'franchise': None, 'min_age': None, 'release_date': None, 'times_sold': 100000, 'url': 'https://www.nintendo.com/us/search/#q=Nintendo+Switch+Pro+Controller', 'store_breakdown': {'Store_A': 40000, 'Store_B': 30000, '

### 4.3 Casos adicionais de QA

In [ ]:
casos = [
    "What accessories go well with the Nintendo Switch?",
    "I want a racing game.",
    "Recommend something similar to Splatoon 3, available at Store C.",
]

for q in casos:
    print(f"\n{'='*70}\nQuery: {q}\n{'='*70}")
    print(run_agent(q, verbose=False))


## 4.5 Modelação dimensional dos dados

Para suportar regras de negócio e escala, os dados brutos podem ser reorganizados num
**esquema em estrela** com chaves substitutas. Isto é demonstrado em `data_model.py`.

Notas:
- `fact_sales` é **agregada** (granularidade = produto × loja), não transacional.
- `fact_cooccurrence` captura a relação entre produtos.

In [15]:
from data_model import build_dimensional_model, check_referential_integrity

model = build_dimensional_model()
warnings = check_referential_integrity(model) 
print('Integridade referencial:', 'OK' if not warnings else warnings)

for nome, df in model.items():
    print(f"\n {nome} ({len(df)} linhas) ")
    display(df.head())


Integridade referencial: OK

 dim_product (16 linhas) 


,product_id,name,category,type,franchise,min_age,release_date,standalone_sales
0,1,Animal Crossing: New Horizons,Game,Simulation,Animal Crossing,3,2020-03-20,400000
1,2,Joy-Con Controllers (Pair),Accessory,Controller,<NA>,<NA>,NaT,20000
2,3,Mario Kart 8 Deluxe,Game,Racing,Super Mario,6,2017-04-28,150000
3,4,Mario Party Superstars,Game,Party,Super Mario,6,2021-10-29,100000
4,5,Nintendo Switch,Console,<NA>,<NA>,<NA>,NaT,50000



 dim_store (3 linhas) 


,store_id,store_name
0,1,Store_A
1,2,Store_B
2,3,Store_C



 fact_sales (44 linhas) 


,product_id,store_id,units_sold
0,5,1,200000
1,5,2,150000
2,5,3,150000
3,15,1,120000
4,15,2,80000



 fact_cooccurrence (111 linhas) 


,product_id_a,product_id_b,times_together
0,5,15,30000
1,5,16,50000
2,5,14,60000
3,5,3,40000
4,5,4,20000


## 5. Limitações

1. **Escala dos dados.** A matriz é 16x16. Com este tamanho, co-ocorrência direta e cosseno
   convergem e são dominados por best-sellers. Em produção (milhares de produtos), seria preciso
   uma base de dados adequada e métodos esparsos.
2. **A diagonal mistura semânticas.** Representa vendas "sozinho", mas é usada no vetor de cada
   produto para o cálculo de cosseno. Surge então decisão a discutir: incluir ou não a diagonal.
3. **Sem lift rigoroso.** A matriz dá contagens de co-ocorrência, mas não o nº total de
   transações nem cabazes com 3+ produtos, por isso não é possível calcular *lift* / *confidence*
   de regras de associação de forma rigorosa sem pressupostos adicionais.

